[Lab README](README.md)

# Lab 2.2: Full-text retrieval, with and without the graph

Embeddings blur exact strings. A postal code, a confirmation number, and a
product SKU all embed to roughly "short numeric token", which is why a question
containing one so often returns chunks about the right topic and the wrong
record.

Full-text search does not have that problem, and it has the opposite one: it
cannot match a paraphrase. Hybrid retrieval runs both and fuses the rankings.

The pair here is `HybridRetriever`, which fuses and returns chunks, against
`HybridCypherRetriever`, which fuses and then traverses. The second one is the
production shape: Lab 3 puts it behind a tool, Lab 4 writes against the
`hotel_id` it returns, and Lab 5 deploys it unchanged.

In [ ]:
# At an AWS event: dependencies are pre-installed. Run this cell as-is.
# Self-paced: uncomment the line below first.
# !pip install -r requirements.txt

print("Environment ready")

## Connect and verify the graph

Retrieval notebooks do not create schema artifacts. This cell requires both Lab 1
indexes to be online with the expected label, property, dimensions, and
similarity function, then checks the graph facts the questions below depend on.

In [ ]:
import os

import boto3
from dotenv import load_dotenv
from IPython.display import HTML, display
from neo4j import GraphDatabase

load_dotenv()

NEO4J_VARS = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")
missing = [name for name in NEO4J_VARS if not os.environ.get(name)]
has_aws = boto3.Session().get_credentials() is not None
RETRIEVAL_READY = not missing and has_aws

if missing:
    print(f"Neo4j is not configured: set {', '.join(missing)} in the repo-root .env")
if not has_aws:
    print("No AWS credentials found, so the query embedding cannot be computed")

if RETRIEVAL_READY:
    # Imported here because `workshop.graph_connection` raises at import when
    # NEO4J_PASSWORD is unset, which would fail this cell instead of skipping it.
    from workshop.graph_connection import NEO4J_URI, neo4j_auth
    from workshop.retrieval_contract import (
        CHUNK_FULLTEXT_INDEX,
        CHUNK_VECTOR_INDEX,
        EMBEDDING_DIMENSIONS,
        EMBEDDING_MODEL_ID,
    )
    from workshop.retrieval_setup import fixture_problems, verify_retrieval_indexes

    driver = GraphDatabase.driver(NEO4J_URI, auth=neo4j_auth())
    driver.verify_connectivity()

    try:
        verify_retrieval_indexes(driver)
        problems = fixture_problems(driver)
        if problems:
            raise RuntimeError("; ".join(problems))
    except Exception as exc:
        raise RuntimeError(
            f"The graph is not ready for retrieval: {exc}\n"
            "Run Lab 1 (01-graph-build/1.1_build_graph.ipynb) first."
        ) from exc

    print(f"{CHUNK_VECTOR_INDEX} is ONLINE")
    print(f"{CHUNK_FULLTEXT_INDEX} is ONLINE")
    print("Every graph fact these questions depend on is present")
else:
    print("\nThe cells below will skip. Finish Lab 0 and Lab 1, then come back.")

## The schema the graph actually holds

Lab 1 pinned this schema during extraction, so the traversals below can name
relationships instead of discovering them. Render it before using any retriever
that traverses, so the shape of the added context is predictable.

In [ ]:
from workshop.graph_schema import GRAPH_SCHEMA

pattern_rows = "".join(
    f"<tr><td><strong>{source}</strong></td><td>&mdash;{relationship}&rarr;</td>"
    f"<td><strong>{target}</strong></td></tr>"
    for source, relationship, target in GRAPH_SCHEMA["patterns"]
)
display(HTML(
    "<table><thead><tr><th>From</th><th>Relationship</th><th>To</th></tr>"
    f"</thead><tbody>{pattern_rows}</tbody></table>"
    "<p>Each extracted entity also points to its source "
    "<code>(entity)-[:FROM_CHUNK]-&gt;(:Chunk)</code>.</p>"
))

## One embedder, shared with the build

A query embedding has to come from the same model, at the same dimension, with
the same purpose as the vectors Lab 1 wrote. A mismatch does not raise. It
returns confident, wrong neighbours. So the embedder is constructed from the
same module the build used rather than configured again here.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from workshop.bedrock_providers import BedrockEmbeddings

    embedder = BedrockEmbeddings(region_name=os.environ.get("AWS_REGION", "us-east-1"))
    print(f"model: {EMBEDDING_MODEL_ID}")
    print(f"dimensions: {EMBEDDING_DIMENSIONS}")

    def show_results(question, result, why):
        """Print each retrieved item with its score, then why the pattern fits."""
        print(f"Question: {question}\n")
        for number, item in enumerate(result.items, 1):
            score = (item.metadata or {}).get("score")
            score_text = "n/a" if score is None else f"{score:.4f}"
            content = str(item.content)
            preview = content[:700] + ("…" if len(content) > 700 else "")
            print(f"[{number}] score={score_text}\n{preview}\n")
        print(f"Why this fits: {why}")

## Pattern 1: hybrid retrieval for an exact identifier

The chunk for Windward Mile Tower contains the postal code `60611`. Ask a
vector-only retriever for the cancellation policy of the hotel at 60611 and it
ranks that chunk poorly, because the digits carry almost no semantic signal.

The cell below runs both. Hybrid retrieval gets the full question's vector for
its semantic arm and the bare postal code for its full-text arm, then fuses the
two rankings. The assertion at the end is the honest part: it fails if the
correct chunk is not in the hybrid results.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from neo4j_graphrag.retrievers import HybridRetriever, VectorRetriever

    vector_retriever = VectorRetriever(
        driver=driver,
        index_name=CHUNK_VECTOR_INDEX,
        embedder=embedder,
        return_properties=["text"],
    )
    identifier_question = "What is the cancellation policy for the hotel at 60611?"
    vector_identifier_result = vector_retriever.search(
        query_text=identifier_question,
        top_k=5,
    )

    hybrid_retriever = HybridRetriever(
        driver=driver,
        vector_index_name=CHUNK_VECTOR_INDEX,
        fulltext_index_name=CHUNK_FULLTEXT_INDEX,
        embedder=embedder,
        return_properties=["text"],
    )
    hybrid_result = hybrid_retriever.search(
        query_text="60611",
        query_vector=vector_identifier_result.metadata["query_vector"],
        top_k=5,
        ranker="linear",
        alpha=0.2,
    )

    show_results(
        identifier_question,
        vector_identifier_result,
        "This is the semantic-only comparison. An exact postal code is not a strong semantic signal.",
    )
    print("\n" + "=" * 80 + "\n")
    show_results(
        identifier_question,
        hybrid_result,
        "Full-text matching preserves 60611 while the separately supplied question vector handles the policy wording.",
    )
    assert any("60611" in str(item.content) for item in hybrid_result.items)

## Pattern 2: the production retriever

`HybridCypherRetriever` is the previous pattern plus the traversal from 2.1. It
is also the one retriever the rest of this workshop uses, so it is not built
inline here. It lives in `workshop.hybrid_retrieval`, and the tool that wraps it
accepts exactly one field:

```python
search_hotel_knowledge(query: str) -> list[HotelEvidence]
```

No ranker, no alpha, no top-k, no retriever-mode selector. Those are the
comparisons you just ran; a caller does not get to re-run them at request time.
Fusion is `NAIVE`, `top_k` is fixed, and the traversal is the same reviewed
Cypher every time.

That narrowness is the point. Lab 3 hands this function to an agent, and an
agent that cannot choose a ranker cannot choose a bad one.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from workshop.contracts import HYBRID_RANKER, HYBRID_TOP_K
    from workshop.graph_setup import HERO_NAME, load_manifest, readiness_problems
    from workshop.hybrid_retrieval import Neo4jConfig, search_hotel_knowledge

    config = Neo4jConfig.from_environment()
    outstanding = readiness_problems(driver, config.database, load_manifest())
    if outstanding:
        raise RuntimeError(
            "This retriever needs the Lab 1 fixtures: "
            + "; ".join(outstanding)
            + "\nRe-run 01-graph-build/1.1_build_graph.ipynb."
        )

    print(f"ranker={HYBRID_RANKER}  top_k={HYBRID_TOP_K}  traversal=fixed")

### The hero question

> **What amenities and guest rating does AnyCompany Cairo Nile View have?**

Every arm of the retriever earns its place on this one question. The exact hotel
name is what the full-text arm is for. "Amenities and guest rating" is a
paraphrase of how the document actually words it, which is what the vector arm
is for. And neither arm returns a rating or an amenity list, because those are
not in the matched chunk. The traversal is what produces them.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    hero_question = f"What amenities and guest rating does {HERO_NAME} have?"
    results = search_hotel_knowledge(hero_question)
    top = results[0]

    print(f"Question: {hero_question}\n")
    print(f"Hotel: {top['hotel_name']}  (hotel_id={top['hotel_id']})")
    print(f"Combined hybrid score: {top['combined_score']:.4f}")
    print(f"Guest rating: {top['guest_rating']}")
    print(f"Exact matched terms: {', '.join(top['exact_terms']) or 'none'}")
    print(f"Amenities ({len(top['amenities'])}): {', '.join(top['amenities'])}")
    print("\nChunk evidence:")
    print(top["chunk_evidence"][:600])

### What just happened

- **Vector matching** handled the paraphrased request for "amenities and guest rating".
- **Full-text matching** locked onto the exact hotel name and location terms.
- The **reviewed Cypher traversal** followed the matched chunk to its hotel and
  returned up to 12 connected amenities, the guest rating, and the opaque
  `hotel_id`. That `hotel_id`, never the display name, is the identity the
  reservation command accepts in Lab 4.

## A question the evidence cannot answer

> **Does AnyCompany Cairo Nile View guarantee room availability next weekend?**

The graph holds hotel knowledge, not live inventory. Below, the same one
retrieval tool goes to a small local agent with the workshop's grounding
instructions. Ask it the hero question, then ask it this one.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    import json

    from strands import Agent, tool
    from strands.models import BedrockModel

    from workshop.hybrid_retrieval import GROUNDING_INSTRUCTIONS

    AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
    MODEL_ID = os.getenv("MODEL_ID", "us.anthropic.claude-sonnet-5")

    @tool
    def search_hotel_knowledge_tool(query: str) -> str:
        """Search grounded hotel evidence and return bounded JSON facts."""
        return json.dumps(search_hotel_knowledge(query), ensure_ascii=False)

    grounded_agent = Agent(
        model=BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION),
        tools=[search_hotel_knowledge_tool],
        system_prompt=(
            "You are a grounded hotel-information assistant. Call "
            "search_hotel_knowledge_tool before answering any hotel question.\n\n"
            + GROUNDING_INSTRUCTIONS
        ),
    )
    print(grounded_agent(hero_question))

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    availability_question = (
        f"Does {HERO_NAME} guarantee room availability next weekend?"
    )
    print(grounded_agent(availability_question))

The retriever returned evidence about the hotel and no evidence about
availability, so the agent said it could not determine the answer instead of
reading "subject to availability" as a yes.

## The mechanism, in one sentence

Vector search finds candidates, traversal finds what is connected.

**Next:** Lab 3 puts `search_hotel_knowledge` behind a Strands `@tool` and gives
it to an agent. `2.3_text2cypher.ipynb` is optional and covers the one question
shape none of these four retrievers handles well: counting.

In [ ]:
if RETRIEVAL_READY:
    driver.close()
    print("Connection closed.")